In [1]:
import pandas as pd
from sklearn.cluster import KMeans
import numpy as np
from ast import literal_eval

In [2]:
def to_entry(ls):
    ret = ''
    for isi in ls:
        ret += ' '
        ret += isi[0]
    return ret[1:]

In [3]:
def rapi(s):
    ret = ''
    for ch in str(s):
        if ch == '-' or ch.isalpha() or ch == ' ':
            ret += ch
    return ret.strip().lower().strip()

In [4]:
def getParallelCorpus(df):
    kalimat_asal_list = []
    kalimat_tujuan_list = []
    kalimat_belum_diproses = []

    total = 0

    split_with_separator = 0
    split_with_font = 0
    split_with_first = 0
    split_with_length = 0

    map = {}
    for i, row in df.iterrows():
        if row['kata_asal'] not in map:
            map[row['kata_asal']] = []
        for kata in eval(row['kata_tujuan']):
            map[row['kata_asal']].append(str(kata))

    for i, row in df.iterrows():
        kalimat = str(row['contoh_kalimat'])
        list_kalimat = str(kalimat).split(',')

        kalimat_lengkap = eval(row['contoh_kalimat_font_size_pos'])
        if(len(kalimat_lengkap) == 0):
            continue

        total += 1


        if(len(list_kalimat) == 2):
            kalimat_asal_list.append(list_kalimat[0].strip())
            kalimat_tujuan_list.append(list_kalimat[1].strip())
            split_with_separator += 1
            continue


        arr = []
        i = 0
        mx_it = -1
        mn_ni = -1
        for kata in kalimat_lengkap:
            if 'italic' in str(kata[1]).lower():
                mx_it = i
            elif mn_ni == -1:
                mn_ni = i
            i += 1

        diff = np.diff(arr)
        indices = np.where(diff != 0)[0] + 1

        if mx_it != -1 and mn_ni != -1 and mx_it < mn_ni:
            idx = mx_it + 1
            split_with_font += 1
            kalimat_asal_list.append(to_entry(kalimat_lengkap[:idx]).strip())
            kalimat_tujuan_list.append(to_entry(kalimat_lengkap[idx:]).strip())
            continue

        found = 0
        if rapi(str(kalimat_lengkap[0][0])) in map:
            tujuan = map[rapi(str(kalimat_lengkap[0][0]))]

            for idx in range(1, len(kalimat_lengkap)):
                if rapi(kalimat_lengkap[idx][0]) in tujuan:
                    kalimat_asal_list.append(to_entry(kalimat_lengkap[:idx]).strip())
                    kalimat_tujuan_list.append(to_entry(kalimat_lengkap[idx:]).strip())
                    split_with_first += 1
                    found = 1
                    break

        if found:
            continue
        idx = len(kalimat_lengkap) // 2
        kalimat_asal_list.append(to_entry(kalimat_lengkap[:idx]).strip())
        kalimat_tujuan_list.append(to_entry(kalimat_lengkap[idx:]).strip())
        split_with_length += 1

    print(str(round(100 * split_with_separator / total, 2)) + '%')
    print(str(round(100 * split_with_font / total, 2)) + '%')
    print(str(round(100 * split_with_first / total, 2)) + '%')
    print(str(round(100 * split_with_length / total, 2)) + '%')

    print('Split with separator = ', split_with_separator, ' (', round(split_with_separator * 100 / total, 2), '%)', sep = '')
    print('Split with font = ', split_with_font, ' (', round(split_with_font * 100 / total, 2), '%)', sep = '')
    print('Split with first = ', split_with_first, ' (', round(split_with_first * 100 / total, 2), '%)', sep = '')
    print('Split with length = ', split_with_length, ' (', round(split_with_length * 100 / total, 2), '%)', sep = '')
    ret = pd.DataFrame()
    ret['kalimat_asal'] = kalimat_asal_list
    ret['kalimat_tujuan'] = kalimat_tujuan_list
    return ret

In [5]:
import os
from pathlib import Path
directory = "../Ekstraksi/"
path = directory + '11. Parallel Corpus/'
cnt = 0
daftar_kamus = [18,68,71,54,89]
for filename in os.listdir(directory + '8. Cleaning Informasi Sumber Daya NLP/'):
    if(filename[-3:] == 'csv'):
        nomor = filename.split('_')[0]
        # if int(nomor) not in daftar_kamus:
            # continue
        f = os.path.join(directory + '8. Cleaning Informasi Sumber Daya NLP/', filename)
        df = pd.read_csv(f)
        df_clean = df
        df_parallel_corpus = getParallelCorpus(df_clean)
        df_parallel_corpus.to_csv(path + nomor + "_Parcor.csv", index = False)
        cnt += 1
        print('Done Kamus', nomor, '(Total', cnt, 'Kamus)')
        # print(nomor, len(df_parallel_corpus), sep = '\t')


82.12%
3.5%
1.29%
13.09%
Split with separator = 2416 (82.12%)
Split with font = 103 (3.5%)
Split with first = 38 (1.29%)
Split with length = 385 (13.09%)
Done Kamus 10 (Total 1 Kamus)
12.97%
1.48%
0.0%
85.55%
Split with separator = 79 (12.97%)
Split with font = 9 (1.48%)
Split with first = 0 (0.0%)
Split with length = 521 (85.55%)
Done Kamus 11 (Total 2 Kamus)
36.88%
6.03%
1.21%
55.88%
Split with separator = 1774 (36.88%)
Split with font = 290 (6.03%)
Split with first = 58 (1.21%)
Split with length = 2688 (55.88%)
Done Kamus 12 (Total 3 Kamus)
7.82%
17.52%
0.17%
74.49%
Split with separator = 46 (7.82%)
Split with font = 103 (17.52%)
Split with first = 1 (0.17%)
Split with length = 438 (74.49%)
Done Kamus 13 (Total 4 Kamus)
6.32%
12.34%
0.0%
81.34%
Split with separator = 63 (6.32%)
Split with font = 123 (12.34%)
Split with first = 0 (0.0%)
Split with length = 811 (81.34%)
Done Kamus 14 (Total 5 Kamus)
37.69%
8.3%
0.14%
53.87%
Split with separator = 268 (37.69%)
Split with font = 59 (8.3